Web Scrapper

In [9]:
# Import Libraries
import requests
import bs4
import pandas as pd
import json
from random import randint
from time import sleep
import re

In [10]:
# Scrape phones from mobileshop.ug via the JSON API its category page loads from
API_URL = "https://mobileshop.ug/api/items"
HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/141.0.0.0 Safari/537.36",
           "Accept": "application/json"}

def spec_text(item, key):
    # Spec fields come as lists of text fragments, e.g. ["Type: 5000 mAh\nCharging: 33W wired", ...]
    return " ".join(item.get(key) or []).replace("\n", " ")

rows = []
page = 1
total_pages = 1
while page <= total_pages:
    response = requests.get(API_URL, params={"category": "mobile-phones", "page": page}, headers=HEADERS, timeout=30)
    response.raise_for_status()
    data = response.json()["data"]
    total_pages = data["totalPages"]
    for item in data["items"]:
        base = {
            "product Names": item["title"].replace(";", "").replace(",", "").replace("[", "").replace("]", "").strip(),
            "Brand": (item.get("subs") or {}).get("name", ""),
            "Condition": item.get("condition", ""),
            "Display": spec_text(item, "display"),
            "Camera": spec_text(item, "camera"),
            "Battery": spec_text(item, "battery"),
        }
        # Each storage option (RAM/ROM) has its own price, so save one row per option
        options = (item.get("variants") or {}).get("storage") or [{}]
        for option in options:
            price = option.get("price") or item.get("discountprice") or item.get("price")
            if not price:
                continue
            rows.append({**base, " Product Prices": int(price), "RAM": option.get("ram", ""), "ROM": option.get("name", "")})
    page = page + 1
    sleep(randint(1, 3))

print(f"Scraped {len(rows)} phone options from {total_pages} pages")

# Saves scraped data to Phones.csv (only if something was scraped, so a failed run doesn't wipe the file)
if rows:
    pd.DataFrame(rows).to_csv("Phones.csv", index=False)

Scraped 606 phone options from 17 pages


#### Load Data and clean the columns

In [11]:
df = pd.read_csv('Phones.csv')
# df.info()

# Drop listings that aren't a phone at its full price: loan deposits and non-phones (e.g. speakers)
not_full_price = df['product Names'].str.contains(r'\bon loan\b', case=False)
not_a_phone = df['product Names'].str.contains(r'\b(?:speaker|power bank)\b', case=False)
df = df[~(not_full_price | not_a_phone)].reset_index(drop=True)

def first_number(text, pattern):
    # First regex match in text as a float, or None
    match = re.search(pattern, str(text), re.IGNORECASE)
    return float(match.group(1)) if match else None

def max_number(text, pattern):
    # Largest regex match in text as a float, or None
    numbers = [float(n) for n in re.findall(pattern, str(text), re.IGNORECASE)]
    return max(numbers) if numbers else None

def to_gb(text):
    # Memory size like "8GB", "1TB", "128MB" or "12" (GB assumed) converted to GB
    match = re.search(r'(\d+(?:\.\d+)?)\s*(TB|GB|G|MB)?', str(text), re.IGNORECASE)
    if not match:
        return None
    size, unit = float(match.group(1)), (match.group(2) or 'GB').upper()
    return size * 1024 if unit == 'TB' else size / 1024 if unit == 'MB' else size

def first_present(*values):
    return next((v for v in values if v is not None and not pd.isna(v)), None)

# Clean the brand column
brand_names = {'Google Pixel': 'Google', 'Nothing Phone': 'Nothing', 'Huawei Honor': 'Honor', 'Zte': 'ZTE'}
brand = df['Brand'].replace(brand_names)

# Screen size in inches: title "6.7-Inch", else display spec "Size: 6.77 inches"
screen_size = [first_present(first_number(title, r'(\d+(?:\.\d+)?)\s*-?\s*Inch'), first_number(display, r'Size:\s*(\d+(?:\.\d+)?)\s*inch'))
               for title, display in zip(df['product Names'], df['Display'])]

# RAM and ROM in GB: storage option, else title "8GB RAM" / "256GB ROM"
ram = [first_present(to_gb(r) if not pd.isna(r) else None, first_number(title, r'(\d+)\s*GB\s*RAM'))
       for r, title in zip(df['RAM'], df['product Names'])]
rom = [first_present(to_gb(r) if not pd.isna(r) else None, first_number(title, r'(\d+)\s*GB\s*(?:ROM|Storage)'))
       for r, title in zip(df['ROM'], df['product Names'])]

# Main camera megapixels: largest "50MP" in the title, else first "50 MP" in the camera spec
mp = [first_present(max_number(title, r'(\d+(?:\.\d+)?)\s*MP'), first_number(camera, r'(\d+(?:\.\d+)?)\s*MP'))
      for title, camera in zip(df['product Names'], df['Camera'])]

# Battery capacity in mAh: title "5000mAh", else battery spec "5000 mAh"
battery = [first_present(first_number(title, r'(\d{3,5})\s*mAh'), first_number(spec, r'(\d{3,5})\s*mAh'))
           for title, spec in zip(df['product Names'], df['Battery'])]

# Release year: longest model-name prefix from release_years.csv that starts the title
# (e.g. "Samsung Galaxy S21 FE" 2022 wins over "Samsung Galaxy S21" 2021)
release_years = pd.read_csv('release_years.csv')
year_keys = sorted(zip(release_years['model'].str.lower(), release_years['release_year']), key=lambda k: -len(k[0]))

def release_year(title):
    title = str(title).lower()
    for model, year in year_keys:
        if title.startswith(model) and (len(title) == len(model) or not title[len(model)].isalnum()):
            return year
    return None

release_year_col = df['product Names'].map(release_year)
unmatched = df.loc[release_year_col.isna(), 'product Names'].unique()
if len(unmatched):
    print(f"{len(unmatched)} models missing from release_years.csv (rows dropped):", *unmatched, sep='\n  ')

# Tier: the model's position within its product line, from keywords in the model name
def model_name(title):
    # Matched release_years.csv model plus any tier words after it, e.g. "iPhone 15" + " Pro Max"
    lower = str(title).lower()
    for model, year in year_keys:
        if lower.startswith(model) and (len(lower) == len(model) or not lower[len(model)].isalnum()):
            rest = re.match(r'((?:\s+(?:pro|max|plus|ultra|mini|fe|lite|edge|xl|fold|flip))*)', str(title)[len(model):], re.IGNORECASE)
            return str(title)[:len(model)] + rest.group(1)
    return str(title)

def phone_tier(model):
    m = model.lower()
    if m.startswith('nokia') or 'xcover' in m:
        return 'Base'  # keypad phones (including flip ones) and rugged Galaxy XCover phones
    galaxy_a = re.match(r'samsung galaxy a(\d)\d', m)
    if galaxy_a:
        # Galaxy A first digit is the tier: A0x/A1x entry, A2x/A3x mid, A5x upper mid
        return {'0': 'Lite', '1': 'Lite', '5': 'Pro'}.get(galaxy_a.group(1), 'Base')
    rules = [
        (r'fold|flip|duo|razr|magic v', 'Foldable'),
        (r'\bultra\b', 'Ultra'),
        (r'\bpro\s*(?:max|xl|plus|\+)', 'Pro Max'),
        (r'\bpro\b', 'Pro'),
        (r'\b(?:plus|max|air)\b|galaxy s\d+ edge', 'Plus'),
        (r'\b(?:fe|lite|mini)\b|\d+a\b', 'Lite'),
    ]
    for pattern, tier in rules:
        if re.search(pattern, m):
            return tier
    return 'Base'

tier_col = df['product Names'].map(model_name).map(phone_tier)

# Product line (series) within the brand, e.g. Galaxy S vs Galaxy A; falls back to the brand's first word
def product_line(model):
    m = model.lower()
    lines = [
        (r'iphone', 'iPhone'), (r'galaxy z', 'Galaxy Z'), (r'galaxy s', 'Galaxy S'), (r'galaxy a', 'Galaxy A'),
        (r'galaxy [fm]', 'Galaxy F/M'), (r'xcover', 'Galaxy XCover'), (r'pixel', 'Pixel'),
        (r'redmi note', 'Redmi Note'), (r'redmi', 'Redmi'), (r'poco', 'Poco'), (r'xiaomi', 'Xiaomi'),
        (r'camon', 'Camon'), (r'spark', 'Spark'), (r'phantom', 'Phantom'), (r'tecno pop', 'Pop'),
        (r'infinix note', 'Infinix Note'), (r'infinix hot', 'Infinix Hot'), (r'infinix smart', 'Infinix Smart'),
        (r'reno', 'Reno'), (r'find', 'Find'), (r'oppo a', 'Oppo A'), (r'nord', 'Nord'), (r'oneplus', 'OnePlus'),
        (r'honor magic', 'Honor Magic'), (r'honor', 'Honor X/Play'), (r'pura', 'Pura'), (r'razr', 'Razr'),
        (r'nubia', 'Nubia'), (r'blade', 'Blade'), (r'realme c', 'Realme C'), (r'realme', 'Realme'),
        (r'nothing', 'Nothing Phone'), (r'nokia', 'Nokia'), (r'vivo', 'Vivo'),
    ]
    for pattern, line in lines:
        if re.search(pattern, m):
            return line
    return model.split()[0].title()

series_col = df['product Names'].map(model_name).map(product_line)

# New DataFrame with cleaned columns
phone_df = pd.DataFrame({
    'brand': brand,
    'screen_size': screen_size,
    'ram': ram,
    'rom': rom,
    'mp': mp,
    'battery': battery,
    'condition': df['Condition'],
    'release_year': release_year_col,
    'series': series_col,
    'tier': tier_col,
    'price': df[' Product Prices'],
})

# Drop rows with missing specs, typo values (e.g. "6TB" RAM) and duplicate colour listings
phone_df = phone_df.dropna()
phone_df = phone_df[(phone_df['ram'] <= 24) & (phone_df['screen_size'].between(1, 10))]
phone_df = phone_df.drop_duplicates().reset_index(drop=True)

# Change the column data types
phone_df['brand'] = phone_df['brand'].astype('category')
phone_df['screen_size'] = phone_df['screen_size'].astype(float)
phone_df['ram'] = phone_df['ram'].astype(float)
phone_df['rom'] = phone_df['rom'].astype(float)
phone_df['mp'] = phone_df['mp'].astype(float)
phone_df['battery'] = phone_df['battery'].astype(int)
phone_df['condition'] = phone_df['condition'].astype('category')
phone_df['release_year'] = phone_df['release_year'].astype(int)
phone_df['series'] = phone_df['series'].astype('category')
phone_df['tier'] = phone_df['tier'].astype('category')
phone_df['price'] = phone_df['price'].astype(int)

# Save clean dataset to a new csv
phone_df.to_csv('phone_df.csv', index=False)
print(f"Cleaned {len(phone_df)} of {len(df)} rows")

Cleaned 581 of 602 rows
